In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import librosa
import numpy as np

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_PATH = os.path.join(BASE_PATH, "genres_stems")

durations = []
sample_rates = []

for genre in os.listdir(TRAIN_PATH):

    genre_path = os.path.join(TRAIN_PATH, genre)

    for song in os.listdir(genre_path):

        song_path = os.path.join(genre_path, song)

        for stem in ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]:

            file_path = os.path.join(song_path, stem)

            audio, sr = librosa.load(file_path, sr=None)

            duration = len(audio) / sr

            durations.append(duration)
            sample_rates.append(sr)

print("Total files checked:", len(durations))
print("Sample rate unique values:", set(sample_rates))
print("Average duration:", np.mean(durations))
print("Minimum duration:", np.min(durations))
print("Maximum duration:", np.max(durations))

Total files checked: 4000
Sample rate unique values: {44100}
Average duration: 30.02404707482993
Minimum duration: 29.931972789115648
Maximum duration: 30.648888888888887


In [3]:
import os
import librosa
import numpy as np

MASHUP_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

durations = []
sample_rates = []

for file in os.listdir(MASHUP_PATH):

    path = os.path.join(MASHUP_PATH, file)

    audio, sr = librosa.load(path, sr=None)

    durations.append(len(audio)/sr)
    sample_rates.append(sr)

print("Total mashups:", len(durations))
print("Sample rate unique values:", set(sample_rates))
print("Average duration:", np.mean(durations))
print("Minimum duration:", np.min(durations))
print("Maximum duration:", np.max(durations))

Total mashups: 3020
Sample rate unique values: {22050}
Average duration: 28.653239927317504
Minimum duration: 6.060408163265306
Maximum duration: 30.57922902494331


In [4]:
import pandas as pd

df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
print(df.head())

   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav


In [5]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [6]:
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_PATH = os.path.join(BASE_PATH,"genres_stems")

genres = sorted(os.listdir(TRAIN_PATH))

label_map = {g:i for i,g in enumerate(genres)}

In [7]:
data = []

for genre in genres:
    
    genre_path = os.path.join(TRAIN_PATH,genre)
    
    for song in os.listdir(genre_path):
        
        song_path = os.path.join(genre_path,song)
        
        stems = {
            "drums": os.path.join(song_path,"drums.wav"),
            "bass": os.path.join(song_path,"bass.wav"),
            "vocals": os.path.join(song_path,"vocals.wav"),
            "other": os.path.join(song_path,"other.wav")
        }
        
        data.append((stems,label_map[genre]))

print("Total songs:",len(data))

Total songs: 1000


In [8]:
train_data, val_data = train_test_split(
    data,
    test_size=0.1,
    random_state=42,
    stratify=[x[1] for x in data]
)

In [9]:
class MashupDataset(Dataset):
    
    def __init__(self,data):
        self.data=data
        self.target_len = 22050 * 30   # 30 seconds
        
    def __len__(self):
        return len(self.data)
    
    def fix_length(self,y):
        
        if len(y) < self.target_len:
            
            pad = self.target_len - len(y)
            y = np.pad(y,(0,pad))
        
        else:
            
            y = y[:self.target_len]
        
        return y
    
    def __getitem__(self,idx):
        
        stems,label=self.data[idx]
        
        audio=[]
        
        for s in stems.values():
            
            y,sr = librosa.load(s,sr=22050)
            
            y = self.fix_length(y)
            
            audio.append(y)
        
        mix=np.sum(audio,axis=0)
        
        mel = librosa.feature.melspectrogram(
            y=mix,
            sr=22050,
            n_mels=128
        )
        
        mel = librosa.power_to_db(mel)
        
        mel = (mel-mel.mean())/(mel.std()+1e-6)
        
        mel = torch.tensor(mel).unsqueeze(0).float()
        
        return mel,label

In [10]:
train_loader=DataLoader(
    MashupDataset(train_data),
    batch_size=8,
    shuffle=True
)

val_loader=DataLoader(
    MashupDataset(val_data),
    batch_size=8
)

In [11]:
class CNNModel(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.conv=nn.Sequential(
            
            nn.Conv2d(1,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        
        self.fc=nn.Linear(64,10)
        
    def forward(self,x):
        
        x=self.conv(x)
        x=x.view(x.size(0),-1)
        return self.fc(x)

In [12]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=CNNModel().to(device)

criterion=nn.CrossEntropyLoss()

optimizer=optim.Adam(model.parameters(),lr=0.001)

In [13]:
EPOCHS=15

for epoch in range(EPOCHS):
    
    model.train()
    
    for x,y in train_loader:
        
        x=x.to(device)
        y=y.to(device)
        
        optimizer.zero_grad()
        
        pred=model(x)
        
        loss=criterion(pred,y)
        
        loss.backward()
        
        optimizer.step()
    
    model.eval()
    
    preds=[]
    targets=[]
    
    with torch.no_grad():
        
        for x,y in val_loader:
            
            x=x.to(device)
            
            out=model(x)
            
            p=torch.argmax(out,1).cpu().numpy()
            
            preds.extend(p)
            targets.extend(y.numpy())
    
    f1=f1_score(targets,preds,average="macro")
    
    print(f"Epoch {epoch+1} F1:",f1)

Epoch 1 F1: 0.06405648267008986
Epoch 2 F1: 0.14477416837381718
Epoch 3 F1: 0.39268341604631923
Epoch 4 F1: 0.29054346554346555
Epoch 5 F1: 0.4417460317460319
Epoch 6 F1: 0.4190014051895867
Epoch 7 F1: 0.4391849529780564
Epoch 8 F1: 0.40095526872148657
Epoch 9 F1: 0.4792825189896129
Epoch 10 F1: 0.5143127331605591
Epoch 11 F1: 0.5012280149518002
Epoch 12 F1: 0.46166170468229967
Epoch 13 F1: 0.66427216058795
Epoch 14 F1: 0.4738847602097029
Epoch 15 F1: 0.6146250028893048


In [14]:
class TestDataset(Dataset):
    
    def __init__(self, df):
        self.df = df
        self.target_len = 22050 * 30
        
    def fix_length(self, y):
        
        if len(y) < self.target_len:
            pad = self.target_len - len(y)
            y = np.pad(y,(0,pad))
        else:
            y = y[:self.target_len]
            
        return y
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        
        path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/" + self.df.iloc[idx]["filename"]
        
        y, sr = librosa.load(path, sr=22050)
        
        y = self.fix_length(y)
        
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=22050,
            n_mels=128
        )
        
        mel = librosa.power_to_db(mel)
        
        mel = (mel-mel.mean())/(mel.std()+1e-6)
        
        mel = torch.tensor(mel).unsqueeze(0).float()
        
        return mel

In [15]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [16]:
test_loader = DataLoader(
    TestDataset(test_df),
    batch_size=8,
    shuffle=False
)

In [17]:
rev_label_map = {v:k for k,v in label_map.items()}

In [18]:
model.eval()

predictions = []

with torch.no_grad():
    
    for x in test_loader:
        
        x = x.to(device)
        
        out = model(x)
        
        pred = torch.argmax(out,1).cpu().numpy()
        
        predictions.extend(pred)

In [19]:
genres_pred = [rev_label_map[p] for p in predictions]

In [20]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": genres_pred
})

submission.to_csv("submission.csv",index=False)

print(submission.head())

   id   genre
0   1  reggae
1   2   blues
2   3  reggae
3   4    rock
4   5    rock
